In [1]:
import anndata
from matplotlib import pyplot as plt
import numpy as np

| Dataset | Paper link | Data link|
|----------|----------|----------|
| ren  | https://www.cell.com/cell/fulltext/S0092-8674(22)00070-8  | https://cellxgene.cziscience.com/collections/8f126edf-5405-4731-8374-b5ce11f53e82 |
| Ren  | https://www.cell.com/cell/fulltext/S0092-8674(21)00148-3  | https://cellxgene.cziscience.com/collections/0a839c4b-10d0-4d64-9272-684c49a2c8ba |
| Schulte-Schrepping | https://www.cell.com/cell/fulltext/S0092-8674(20)30992-2| Not available |
| ren | https://doi.org/10.1038/s41591-021-01329-2 | https://cellxgene.cziscience.com/collections/ddfad306-714d-4cc0-9985-d9072820c530 | 
| Wilk | https://rupress.org/jem/article/218/8/e20210582/212379/Multi-omic-profiling-reveals-widespread | https://www.covid19cellatlas.org/index.patient.html, search Blish, second option PMBCs and Granulocytes |

#### High level description of this notebook

In each of these datasets, we need to put labels in a common format. They need the following labels:
- `donor_id`: unique identifier for each donor
- `label`: severity of COVID-19 infection, e.g. 'Mild', 
- `cell_type`: cell type, e.g. 'Granulocytes', 'Monocytes', 'B cells', etc.


### Helpers

In [2]:
def label_map(anndata, column_name, mapping_dict):
    """ """
    def mapper(row):
        value = row[column_name]
        for key, mapped_value in mapping_dict.items():
            if value in key:
                return mapped_value
        return np.nan
    
    anndata.obs['label'] = anndata.obs.apply(mapper, axis=1)

    # remove samples not in the mapping dict
    anndata = anndata[~anndata.obs['label'].isna()]
    return anndata

def remove_low_count_donors(anndata):
    donor_counts =  anndata.obs['donor_id'].value_counts()
    return anndata[anndata.obs['donor_id'].isin(donor_counts[donor_counts >= 1000].index)]


### Combat Dataset

In [2]:
combat = anndata.read_h5ad("combat.h5ad")
combat

AnnData object with n_obs × n_vars = 836148 × 37298
    obs: 'GEX_region', 'cluster', 'cluster_source', 'minor_subset', 'minor_subset_source', 'major_subset', 'major_subset_source', 'cell_type_source', 'scRNASeq_sample_ID', 'QC_ngenes', 'QC_total_UMI', 'QC_pct_mitochondrial', 'QC_scrub_doublet_scores', 'TCR_clone_count', 'TCR_clone_proportion', 'BCR_total_mut_HC', 'BCR_clonal_abundance_HC', 'BCR_total_mut_LC', 'assay_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'cell_type_original', 'donor_id', 'Source', 'Age', 'BMI', 'Hospitalstay', 'Death28', 'Institute', 'PreExistingHeartDisease', 'PreExistingLungDisease', 'PreExistingKidneyDisease', 'PreExistingDiabetes', 'PreExistingHypertension', 'PreExistingImmunocompromised', 'Smoking', 'Symptomatic', 'Requiredvasoactive', 'SARSCoV2PCR', 'Outcome', 'TimeSinceOnset', 'sex_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'organism_ontolo

"Following inspection of the QC metrics, the dataset was filtered to retain cells with ngenes > 300 and pct_mitochondrial < 10%. In total, n = 836,148 cells were selected for downstream analysis"

I have confirmed this data is post-QC.

In [4]:

combat_mapping = {
    ('COVID_SEV', 'COVID_CRIT'): 'severe',
    ('COVID_MILD', 'COVID_HCW_MILD'): 'mild',
    ('HV',): 'control'
}

combat = label_map(combat, 'Source', combat_mapping)
combat = remove_low_count_donors(combat)

# Get unique donor count per label
unique_donors_per_label = combat.obs.groupby('label')['donor_id'].nunique()
print(unique_donors_per_label)


combat.obs['donor_id'].value_counts()



label
control    10
mild       29
severe     46
Name: donor_id, dtype: int64


donor_id
S00052    21654
S00033    17477
S00045    16767
S00005    16259
S00109    16190
          ...  
S00095     3245
S00027     3220
S00077     2995
S00099     2522
S00008     1657
Name: count, Length: 85, dtype: int64

In [5]:
combat.write_h5ad("combat_processed.h5ad")

/home/benjami/micromamba/envs/geneformer2/lib/python3.11/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


### Ren Dataset

In [3]:
ren = anndata.read_h5ad("ren.h5ad")
ren

AnnData object with n_obs × n_vars = 1462702 × 27647
    obs: 'celltype', 'majorType', 'City', 'sampleID', 'donor_id', 'Sample type', 'CoVID-19 severity', 'Sample time', 'Sampling day (Days after symptom onset)', 'BCR single cell sequencing', 'TCR single cell sequencing', 'Outcome', 'Comorbidities', 'COVID-19-related medication and anti-microbials', 'Leukocytes [G over L]', 'Neutrophils [G over L]', 'Lymphocytes [G over L]', 'Unpublished', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    u

In [4]:
ren.obs['CoVID-19 severity'].value_counts()

CoVID-19 severity
mild/moderate      700968
severe/critical    596734
control            165000
Name: count, dtype: int64

In [6]:

ren_mapping = {
    ('severe/critical'): 'severe',
    ('mild/moderate'): 'mild',
    ('control',): 'control'
}

ren = label_map(ren, 'CoVID-19 severity', ren_mapping)
ren = remove_low_count_donors(ren)

# Get unique donor count per label
unique_donors_per_label = ren.obs.groupby('label')['donor_id'].nunique()
print(unique_donors_per_label)


ren.obs['donor_id'].value_counts()



/tmp/ipykernel_1412312/4293305759.py:10: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  anndata.obs['label'] = anndata.obs.apply(mapper, axis=1)


label
control    25
mild       77
severe     83
Name: donor_id, dtype: int64


donor_id
P-M004    49223
P-M007    31763
P-M010    31359
P-S022    29408
P-S086    28625
          ...  
P-S071     1374
P-S005     1356
P-S091     1246
P-S010     1170
P-M056     1024
Name: count, Length: 185, dtype: int64

In [7]:
ren.write_h5ad("ren_processed.h5ad")

/home/benjami/micromamba/envs/geneformer2/lib/python3.11/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


### Stevenson

In [8]:
stevenson = anndata.read_h5ad("stevenson.h5ad")
stevenson

AnnData object with n_obs × n_vars = 647366 × 24677
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'initial_clustering', 'Resample', 'Collection_Day', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'author_cell_type', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'antibody_X', 'ant

In [9]:
stevenson.obs['Status_on_day_collection_summary'].value_counts()

Status_on_day_collection_summary
Moderate        198632
Mild            135936
Healthy          97039
Critical         80852
Severe           78265
Asymptomatic     33601
Non_covid        15157
LPS_90mins        3999
LPS_10hours       3885
Name: count, dtype: int64

In [ ]:

stevenson_mapping = {
    ('Mild', 'Moderate'): 'mild',
    ('Severe', 'Critical'): 'severe',
    ('Healthy',): 'control'
}

stevenson = label_map(stevenson, 'Status_on_day_collection_summary', stevenson_mapping)

# Use sample_id as donor_id: some donors were sampled at multiple timepoints
# with different severity statuses (e.g. Moderate on D0, Critical on D12).
# Since we're classifying current blood state, each sample is an independent unit.
stevenson.obs['donor_id'] = stevenson.obs['sample_id']

stevenson = remove_low_count_donors(stevenson)

# Get unique donor count per label
unique_donors_per_label = stevenson.obs.groupby('label')['donor_id'].nunique()
print(unique_donors_per_label)


stevenson.obs['donor_id'].value_counts()

In [11]:
stevenson.write_h5ad("stevenson_processed.h5ad")

/home/benjami/micromamba/envs/geneformer2/lib/python3.11/site-packages/anndata/_core/anndata.py:1145: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


## Cell type labels

In [ ]:
stevenson.obs['cell_type'].unique()

['effector CD8-positive, alpha-beta T cell', 'T-helper 22 cell', 'naive thymus-derived CD8-positive, alpha-beta..., 'naive thymus-derived CD4-positive, alpha-beta..., 'effector memory CD8-positive, alpha-beta T cell', ..., 'hematopoietic precursor cell', 'myeloid lineage restricted progenitor cell', 'megakaryocyte', 'T-helper 17 cell', 'malignant cell']
Length: 46
Categories (46, object): ['erythrocyte', 'platelet', 'B cell', 'dendritic cell', ..., 'group 2 innate lymphoid cell, human', 'T follicular helper cell', 'CD14-low, CD16-positive monocyte', 'hematopoietic precursor cell']

In [ ]:
ren.obs['cell_type'].unique()

['CD14-positive, CD16-negative classical monocyte', 'B cell', 'macrophage', 'mature alpha-beta T cell', 'monocyte', ..., 'ciliated cell', 'squamous epithelial cell', 'secretory cell', 'epithelial cell', 'mast cell']
Length: 18
Categories (18, object): ['ciliated cell', 'epithelial cell', 'squamous epithelial cell', 'mast cell', ..., 'mature alpha-beta T cell', 'mature gamma-delta T cell', 'CD14-positive, CD16-negative classical monocyte', 'CD14-positive, CD16-positive monocyte']

In [ ]:
combat.obs['cell_type'].unique()

['natural killer cell', 'CD8-positive, alpha-beta T cell', 'blood cell', 'non-classical monocyte', 'classical monocyte', ..., 'hematopoietic stem cell', 'dendritic cell', 'megakaryocyte-erythroid progenitor cell', 'enucleated reticulocyte', 'mast cell']
Length: 17
Categories (17, object): ['hematopoietic stem cell', 'megakaryocyte-erythroid progenitor cell', 'blood cell', 'mast cell', ..., 'mucosal invariant T cell', 'plasmablast', 'enucleated reticulocyte', 'double negative T regulatory cell']